# Question 3 — Grayscale image enhancement

**Input:** grayscale image `image_f` (use `image_f.png` by default).

The notebook computes minimum, maximum, mean, standard deviation and median gray levels; determines whether the histogram is narrow or widely spread; then performs:
- Linear stretching: `round(255 × (r − r_min) / (r_max − r_min))`
- Log stretching: `round(255 × log(1 + r − r_min) / log(1 + r_max − r_min))`
- Histogram equalization using the cumulative histogram.
- Laplacian sharpening with the 4-neighbour kernel (centre −4, four neighbours +1), followed by `s = clip(round(f − L), 0, 255)`.

Two implementation styles are included.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_PATH="image_f.png"

def read_gray(path):
    a=cv2.imread(path,0)
    if a is None: raise FileNotFoundError(f"Cannot read {path}; place image_f.png beside this notebook.")
    return a

def show_gray(title,a):
    plt.figure(figsize=(6,5));plt.imshow(a,cmap="gray",vmin=0,vmax=255);plt.title(title);plt.axis("off");plt.show()

def show_hist(title,a):
    plt.figure(figsize=(7,3));plt.hist(a.ravel(),bins=256,range=(0,256));plt.title(title)
    plt.xlabel("Gray level");plt.ylabel("Frequency");plt.show()

## Style 1 — OpenCV built-in methods

In [ ]:
img1 = read_gray(IMAGE_PATH)
rmin1, rmax1 = int(img1.min()), int(img1.max())
mean1 = float(img1.mean())
std1 = float(img1.std())
median1 = float(np.median(img1))
print(f"Minimum: {rmin1}")
print(f"Maximum: {rmax1}")
print(f"Mean: {mean1:.2f}")
print(f"Standard deviation: {std1:.2f}")
print(f"Median: {median1:.2f}")
print(
    "Histogram assessment:",
    "narrow / low contrast" if rmax1 - rmin1 < 128 else "widely spread",
    f"(range = {rmax1-rmin1})",
)

# Linear stretch using the actual min and max.
if rmax1 == rmin1:
    linear1 = np.zeros_like(img1)
else:
    linear1 = (
        np.rint(255 * (img1.astype(float) - rmin1) / (rmax1 - rmin1))
        .clip(0, 255)
        .astype(np.uint8)
    )

# Logarithmic stretch as specified.
den = np.log(1 + rmax1 - rmin1) if rmax1 != rmin1 else 1
log1 = (
    np.rint(255 * np.log1p(img1.astype(float) - rmin1) / den)
    .clip(0, 255)
    .astype(np.uint8)
)

# Histogram equalization.
equal1 = cv2.equalizeHist(img1)

# 4-neighbour Laplacian and sharpening: sharpened = f - L.
lap_kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)
lap1 = cv2.filter2D(img1, cv2.CV_32F, lap_kernel, borderType=cv2.BORDER_REPLICATE)
sharp1 = np.clip(np.rint(img1.astype(float) - lap1), 0, 255).astype(np.uint8)

for name, a in [
    ("Original", img1),
    ("Linear stretch", linear1),
    ("Log stretch", log1),
    ("Equalized", equal1),
]:
    show_gray(name, a)
    show_hist(name + " histogram", a)
show_gray(
    "Laplacian response (display scaled)", np.uint8(np.clip(np.abs(lap1) * 4, 0, 255))
)
show_gray("Sharpened image", sharp1)

[ WARN:0@0.270] global loadsave.cpp:278 findDecoder imread_('image_f.png'): can't open/read file: check file path/integrity


FileNotFoundError: Cannot read image_f.png; place image_f.png beside this notebook.

## Style 2 — NumPy implementation (OpenCV only for `imread`)

All transformations and the Laplacian convolution are calculated manually. NumPy is used for array arithmetic and histogram accumulation. Matplotlib is used only for image and histogram display.

In [ ]:
# cv2.imread is the ONLY OpenCV operation in this section.
img2=cv2.imread(IMAGE_PATH,0)
if img2 is None: raise FileNotFoundError(f"Cannot read {IMAGE_PATH}")
img2=img2.astype(np.uint8)

rmin2=int(np.min(img2));rmax2=int(np.max(img2))
mean2=float(np.mean(img2));std2=float(np.std(img2));median2=float(np.median(img2))
print(f"Minimum: {rmin2}")
print(f"Maximum: {rmax2}")
print(f"Mean: {mean2:.2f}")
print(f"Standard deviation: {std2:.2f}")
print(f"Median: {median2:.2f}")
print("Histogram assessment:", "narrow / low contrast" if rmax2-rmin2 < 128 else "widely spread",
      f"(range = {rmax2-rmin2})")

if rmax2==rmin2: linear2=np.zeros_like(img2)
else: linear2=np.rint(255*(img2.astype(float)-rmin2)/(rmax2-rmin2)).clip(0,255).astype(np.uint8)

den2=np.log(1+rmax2-rmin2) if rmax2!=rmin2 else 1
log2=np.rint(255*np.log1p(img2.astype(float)-rmin2)/den2).clip(0,255).astype(np.uint8)

hist2=np.bincount(img2.ravel(),minlength=256)
cdf2=np.cumsum(hist2)
cdf_min2=cdf2[np.nonzero(hist2)[0][0]]
if img2.size==cdf_min2: equal2=img2.copy()
else:
    lut2=np.rint((cdf2-cdf_min2)*255/(img2.size-cdf_min2)).clip(0,255).astype(np.uint8)
    equal2=lut2[img2]

def convolve_replicate(a,kernel):
    a=a.astype(float); kh,kw=kernel.shape; py,px=kh//2,kw//2
    p=np.pad(a,((py,py),(px,px)),mode="edge")
    out=np.zeros(a.shape,float)
    for i in range(kh):
        for j in range(kw):
            out += kernel[i,j]*p[i:i+a.shape[0],j:j+a.shape[1]]
    return out

lap_kernel2=np.array([[0,1,0],[1,-4,1],[0,1,0]],float)
lap2=convolve_replicate(img2,lap_kernel2)
sharp2=np.clip(np.rint(img2.astype(float)-lap2),0,255).astype(np.uint8)

for name,a in [("Original",img2),("Linear stretch",linear2),("Log stretch",log2),("Equalized",equal2)]:
    show_gray(name,a);show_hist(name+" histogram",a)
show_gray("Laplacian response (absolute, scaled)",np.uint8(np.clip(np.abs(lap2)*4,0,255)))
show_gray("Sharpened image",sharp2)

### Notes
- Histogram spread is judged here by the observed gray-level range (`max − min`); the threshold of 128 is a simple descriptive rule, not a universal definition.
- The Laplacian sign convention is important. This notebook uses the question's stated sharpening equation `f − L` with the kernel whose centre is −4 and four neighbours are +1.
- If all pixels have the same value, linear/log stretching safely returns a constant black image rather than dividing by zero.